# 05 — Price model: training on A + B

Reads `workspace.gold.listing_features` for the training cities and writes:
- `workspace.gold.price_predictions` — a prediction for **every** listing (both baseline and model);
- `workspace.gold.model_metrics` — test-set MAE per city and for both cities together;
- `workspace.gold.model_feature_importance` — how much MAE grows when a feature is shuffled;
- the model file `price_model_<version>.joblib` in the volume `workspace.gold.models`.

**After this notebook the model is frozen.** The first run on city C uses exactly this file, via `06_score_city`.
A new model version means a new `model_version` (e.g. `v2`); the old file is never overwritten.

**Model versions** (`model_spec`, defined in `model_utils.SPECS`):
- `v1` — week 1 model, used for the first run on city C;
- `v2` — after the city C run: density relative to the city, price relative to the city's price level (see `MODEL.md`).

Shared code (features, split, baseline, pipeline) lives in `model_utils.py` next to this notebook.

In [0]:
dbutils.widgets.text("train_cities", "rome,sicily")
dbutils.widgets.text("model_version", "v2")
dbutils.widgets.dropdown("model_spec", "v2", ["v1", "v2"])
dbutils.widgets.text("model_dir", "/Volumes/workspace/gold/models")
dbutils.widgets.dropdown("overwrite_model", "no", ["no", "yes"])

TRAIN_CITIES = [c.strip().lower() for c in dbutils.widgets.get("train_cities").split(",") if c.strip()]
MODEL_VERSION = dbutils.widgets.get("model_version").strip()
MODEL_DIR = dbutils.widgets.get("model_dir").rstrip("/")
OVERWRITE = dbutils.widgets.get("overwrite_model") == "yes"
SPEC = dbutils.widgets.get("model_spec")
RUN_LABEL = "train_test"

In [0]:
import os, sys, json, datetime
import numpy as np, pandas as pd, sklearn, joblib
from pyspark.sql import functions as F

sys.path.append(os.getcwd())          # model_utils.py sits next to the notebook
import model_utils as mu

CATALOG = "workspace"
FEATURES = f"{CATALOG}.gold.listing_features"
PREDICTIONS = f"{CATALOG}.gold.price_predictions"
METRICS = f"{CATALOG}.gold.model_metrics"
IMPORTANCE = f"{CATALOG}.gold.model_feature_importance"
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.gold.models")

model_path = f"{MODEL_DIR}/price_model_{MODEL_VERSION}.joblib"
if os.path.exists(model_path) and not OVERWRITE:
    raise RuntimeError(f"{model_path} already exists. The model is frozen: use a new model_version for changes "
                       f"or explicitly set overwrite_model = yes")

## 1. Data and split

Only `in_model_scope` rows are used for training and evaluation (price present, `outlier_class` ∈ {normal, tail}).
Test = ~20% of the listings of **each city**, assigned by whole hosts (`host_id`), seed 42. The split is made before any model choice,
and every number below comes from the test set only.

In [0]:
pdf = spark.table(FEATURES).filter(F.col("city").isin(TRAIN_CITIES)).toPandas()
missing = set(TRAIN_CITIES) - set(pdf["city"].unique())
if missing:
    raise ValueError(f"No features for {missing} — run 04_gold_features for these cities")

pdf = mu.add_relative_density(pdf)              # v2 feature; relative to ALL listings of the city
pdf["in_model_scope"] = mu.in_scope(pdf)
scope = pdf[pdf["in_model_scope"]].copy()
scope["split"] = mu.host_split(scope)
pdf["split"] = scope["split"].reindex(pdf.index).fillna("out_of_scope")

display(pdf.groupby(["city", "split"]).size().unstack(fill_value=0).reset_index())
train, test = scope[scope.split == "train"], scope[scope.split == "test"]

city,out_of_scope,test,train
rome,3565,6777,27109
sicily,6075,10129,40579


## 2. Baseline and model

- **Baseline** — median price for the same (`room_type`, `accommodates`) on the training split of **the same city**; if the combination is not in train → median for the `room_type` → city median.
  For reference we also compute `baseline_ab`: the same rule fitted on the training data of both cities together (this is the "old rule" we would apply to city C).
- **Model** — gradient boosting (`HistGradientBoostingRegressor`) on `log(price)`; predictions are converted back with `exp()`.
  Features: size, property and room type, amenities, host profile, long-stay, licence, and the density of surrounding listings (instead of coordinates and neighbourhood names, which do not transfer to a new city). The city itself is not a feature.
- **v2 changes** (after the first run on city C): density is measured relative to the city's own median density, and the target is
  `log(price) − log(median price of the city's train split)`. The model learns how far a listing is above or below its city's price level;
  the level of a new city comes from that city's own training split.

In [0]:
city_baselines = {c: mu.fit_baseline(g) for c, g in train.groupby("city")}
baseline_ab = mu.fit_baseline(train)
levels = mu.fit_city_level(train) if mu.SPECS[SPEC]["city_level"] else None
model = mu.fit_model(train, SPEC, levels)
if levels:
    print("City price levels (median price of train):", {c: round(float(np.exp(v)), 1) for c, v in levels.items()})

pdf["baseline_pred"] = np.nan
for c, b in city_baselines.items():
    m = pdf["city"] == c
    pdf.loc[m, "baseline_pred"] = mu.predict_baseline(b, pdf[m])
pdf["baseline_ab_pred"] = mu.predict_baseline(baseline_ab, pdf)
pdf["model_pred"] = mu.predict_model(model, pdf, SPEC, levels)

City price levels (median price of train): {'rome': 188.0, 'sicily': 115.0}


## 3. Test-set MAE: each city and both together

In [0]:
t = pdf[pdf["split"] == "test"]
rows = [{"city": c, **mu.metrics_rows(g)} for c, g in t.groupby("city")]
rows.append({"city": "all", **mu.metrics_rows(t)})
metrics = pd.DataFrame(rows)
display(metrics[["city", "n_test", "median_price", "mae_baseline", "mae_model", "improvement_vs_baseline_pct",
                 "median_ae_model", "bias_model", "mape_model_pct"]])

city,n_test,median_price,mae_baseline,mae_model,improvement_vs_baseline_pct,median_ae_model,bias_model,mape_model_pct
rome,6777,189.6,84.02,64.19,23.6,35.47,-16.07,26.4
sicily,10129,115.15,73.3,58.99,19.5,28.39,-21.97,33.1
all,16906,142.33,77.59,61.07,21.3,31.04,-19.6,30.4


In [0]:
# Error by room_type — to see where the model is weaker
t2 = t.assign(abs_err_model=(t.model_pred - t.price).abs(), abs_err_baseline=(t.baseline_pred - t.price).abs())
display(t2.groupby(["city", "room_type"]).agg(n=("price", "size"), median_price=("price", "median"),
                                              mae_baseline=("abs_err_baseline", "mean"), mae_model=("abs_err_model", "mean"))
          .round(1).reset_index())

city,room_type,n,median_price,mae_baseline,mae_model
rome,Entire home/apt,5150,208.7,92.8,69.9
rome,Hotel room,30,210.2,62.2,70.6
rome,Private room,1588,136.1,56.3,45.9
rome,Shared room,9,25.9,24.7,16.5
sicily,Entire home/apt,7974,124.0,80.3,64.0
sicily,Hotel room,21,186.0,84.4,75.3
sicily,Private room,2130,92.0,47.2,40.3
sicily,Shared room,4,33.0,24.5,27.0


## 4. Which features matter

Permutation importance on the test set (5,000 listings, 3 repeats): shuffle one feature and measure how many EUR the MAE grows.
This is the model explanation for the report and the presentation.

In [0]:
rng = np.random.default_rng(mu.SEED)
sample = t.sample(min(5000, len(t)), random_state=mu.SEED)
base_mae = np.mean(np.abs(mu.predict_model(model, sample, SPEC, levels) - sample.price))
imp = []
for feat in mu.features_of(SPEC):
    deltas = []
    for _ in range(3):
        sh = sample.copy()
        sh[feat] = rng.permutation(sh[feat].values)
        deltas.append(np.mean(np.abs(mu.predict_model(model, sh, SPEC, levels) - sh.price)) - base_mae)
    imp.append({"feature": feat, "mae_increase_eur": round(float(np.mean(deltas)), 2)})
importance = pd.DataFrame(imp).sort_values("mae_increase_eur", ascending=False)
display(importance)

feature,mae_increase_eur
log_density_5km_rel,8.19
accommodates,6.49
bathrooms,5.66
am_pool,4.6
bedrooms,4.49
log_density_1km_rel,4.1
property_type,2.18
long_stay,2.0
host_tenure_months,1.89
calculated_host_listings_count_entire_homes,1.88


## 5. Save the model and write the gold tables

In [0]:
artifact = {
    "model": model, "baseline_ab": baseline_ab, "city_baselines": city_baselines,
    "spec": SPEC, "city_levels": levels,
    "features": mu.features_of(SPEC), "train_cities": TRAIN_CITIES, "model_version": MODEL_VERSION,
    "sklearn_version": sklearn.__version__, "trained_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "train_rows": int(len(train)), "test_metrics": metrics.to_dict("records"),
}
joblib.dump(artifact, model_path)
print("Saved:", model_path)

Saved: /Volumes/workspace/gold/models/price_model_v2.joblib


In [0]:
def save(pdf_out, table, where):
    sdf = spark.createDataFrame(pdf_out)
    w = sdf.write.format("delta").option("mergeSchema", "true")
    if spark.catalog.tableExists(table):
        w.mode("overwrite").option("replaceWhere", where).saveAsTable(table)
    else:
        w.mode("overwrite").partitionBy("city").saveAsTable(table)

def prediction_table(pdf):
    p = pdf[["city", "snapshot_date", "id", "host_id", "name", "listing_url", "neighbourhood_cleansed", "latitude", "longitude",
             "room_type", "property_type", "accommodates", "bedrooms", "in_model_scope", "split", "outlier_class",
             "price", "baseline_pred", "baseline_ab_pred", "model_pred"]].copy()
    p["error"] = p["model_pred"] - p["price"]                  # < 0: the model under-prices
    p["abs_error"] = p["error"].abs()
    p["pct_error"] = p["error"] / p["price"]
    p["model_vs_actual"] = np.select(
        [p["price"].isna(), p["pct_error"] < -0.2, p["pct_error"] > 0.2],
        ["no actual price", "under-priced (model < actual by >20%)", "over-priced (model > actual by >20%)"],
        default="within 20%")
    p["model_version"], p["run_label"] = MODEL_VERSION, RUN_LABEL
    p["snapshot_date"] = pd.to_datetime(p["snapshot_date"]).dt.date
    for c in ["price", "baseline_pred", "baseline_ab_pred", "model_pred", "error", "abs_error", "pct_error", "latitude",
              "longitude", "accommodates", "bedrooms"]:
        p[c] = p[c].astype("float64")
    p["in_model_scope"] = p["in_model_scope"].astype(bool)
    for c in ["name", "listing_url", "neighbourhood_cleansed", "room_type", "property_type", "split", "outlier_class"]:
        p[c] = p[c].astype(object).where(p[c].notna(), None)
    return p

cities_sql = ", ".join(f"'{c}'" for c in TRAIN_CITIES + ["all"])
where = f"model_version = '{MODEL_VERSION}' AND run_label = '{RUN_LABEL}' AND city IN ({cities_sql})"

save(prediction_table(pdf), PREDICTIONS, where)
m_out = metrics.assign(model_version=MODEL_VERSION, run_label=RUN_LABEL, eval_set="test",
                       train_cities=",".join(TRAIN_CITIES), created_at=datetime.datetime.now())
save(m_out, METRICS, where)
imp_out = importance.assign(city="all", model_version=MODEL_VERSION, run_label=RUN_LABEL)
save(imp_out, IMPORTANCE, where)

In [0]:
display(spark.table(PREDICTIONS).filter(F.col("model_version") == MODEL_VERSION)
        .groupBy("run_label", "city", "split").count().orderBy("city", "split"))
display(spark.table(METRICS).filter(F.col("model_version") == MODEL_VERSION))

run_label,city,split,count
train_test,rome,out_of_scope,3565
train_test,rome,test,6777
train_test,rome,train,27109
train_test,sicily,out_of_scope,6075
train_test,sicily,test,10129
train_test,sicily,train,40579


city,n_test,median_price,mae_baseline,mae_baseline_ab,mae_model,improvement_vs_baseline_pct,median_ae_model,bias_model,mape_model_pct,model_version,run_label,eval_set,train_cities,created_at
sicily,10129,115.15,73.3,77.82,58.99,19.5,28.39,-21.97,33.1,v2,train_test,test,"rome,sicily",2026-11-11T21:40:48.654Z
rome,6777,189.6,84.02,98.67,64.19,23.6,35.47,-16.07,26.4,v2,train_test,test,"rome,sicily",2026-11-11T21:40:48.654Z
all,16906,142.33,77.59,86.18,61.07,21.3,31.04,-19.6,30.4,v2,train_test,test,"rome,sicily",2026-11-11T21:40:48.654Z


In [0]:
display(spark.table("workspace.gold.model_metrics")
        .filter("run_label = 'train_test'")
        .select("model_version", "city", "mae_baseline", "mae_model", "improvement_vs_baseline_pct")
        .orderBy("city", "model_version"))

model_version,city,mae_baseline,mae_model,improvement_vs_baseline_pct
v1,all,77.59,61.23,21.1
v2,all,77.59,61.07,21.3
v1,rome,84.02,63.99,23.8
v2,rome,84.02,64.19,23.6
v1,sicily,73.3,59.38,19.0
v2,sicily,73.3,58.99,19.5
